# Pipeline de extracción de ratios financieros — Imagen

Notebook correspondiente al **experimento de extracción desde imagen** descrito en la sección 4.3 de la memoria del TFM.

El objetivo es extraer las mismas 22 partidas contables y calcular los mismos ratios que en el pipeline Excel, pero cuando el estado financiero llega en formato imagen (PNG, JPEG), ya sea una exportación directa del Excel, una captura de pantalla o una fotografía de un documento impreso.

## Empresa de prueba
**Grifols, S.A.** — imagen generada directamente a partir del Excel de SABI que comprime balance y cuenta de PyG de los tres ejercicios (2022, 2023, 2024) en un único plano visual.

## El problema de la densidad

Una imagen que contiene balance completo y PyG de tres ejercicios tiene una densidad de filas tan alta que Docling fusiona filas adyacentes en una misma celda. La separación visual entre filas es inferior al umbral que el modelo de layout de Docling necesita para identificarlas como entidades separadas.

**Solución:** segmentar la imagen en trozos horizontales con solapamiento del 5% antes de pasarlos a Docling, reduciendo la densidad hasta que la separación es suficiente. El experimento sistemático que justifica el número óptimo de recortes está en `experimentos_recortes.ipynb`.

## Pipeline completo

```
Imagen (.png / .jpg)
    │
    ├─ Recorte en N trozos con overlap 5%      ← PIL
    │
    ├─ Extracción estructurada de cada trozo   ← Docling + RapidOCR
    │
    ├─ Detección de fechas canónicas           ← LLM multimodal (solo cabecera)
    │
    ├─ Detección de layout por trozo           ← LLM textual (muestra serializada)
    │
    ├─ DataFrame de conceptos                  ← Python determinista
    │
    ├─ Selección de 22 partidas                ← LLM (índice posicional)
    │
    ├─ Extracción de valores                   ← Python determinista
    │
    └─ Cálculo de 22 ratios                    ← Python determinista
```

A partir del paso de construcción del DataFrame, la lógica es **idéntica al pipeline Excel** y produce resultados equivalentes.

---


## Celda 1 — Setup, recorte de imagen y extracción con Docling

Esta celda hace tres cosas:

**1. Configuración del entorno**
API keys desde variables de entorno (`.env`), modelos Groq y directorios de entrada/salida. Se definen tres modelos: `MODELO1` y `MODELO2` para llamadas de texto y `MODELO3` (`llama-4-scout-17b`) para la llamada multimodal de extracción de fechas.

**2. Función `recortar_y_escalar`**
Divide la imagen en un fragmento definido por `(top_pct, bottom_pct)` añadiendo un solapamiento del 5% en cada extremo (`overlap_pct=0.05`). El overlap garantiza que ninguna fila quede cortada entre dos trozos consecutivos. Aplica escalado ×2 con `Image.LANCZOS` antes de guardar para mejorar la resolución que recibe el OCR.

**3. Pipeline de extracción**
Para cada imagen en la carpeta de inputs genera 4 trozos (cuartos: 0-25%, 25-50%, 50-75%, 75-100%), pasa cada trozo por `DocumentConverter` de Docling y acumula los DataFrames resultantes en `dfs_trozos`.

> **Por qué 4 trozos:** el experimento en `experimentos_recortes.ipynb` demuestra que con esta imagen concreta 4 recortes es el número a partir del cual Docling deja de fusionar filas. El número óptimo depende de la densidad de filas de cada imagen y no es universal.


In [4]:
#En esta celda se hace la lectura de las imagenes y se recortar cada una en tres partes porque sino las celdas son muy pequeñas y docling junta algunas filas.

from PIL import Image
import pandas as pd
import json
import os

GROQ_API_KEY1  = os.getenv("GROQ_API_KEY_1", "")
GROQ_API_KEY2  = os.getenv("GROQ_API_KEY_2", "")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
MODELO1        = "openai/gpt-oss-120b"
MODELO2        = "llama-3.3-70b-versatile"
MODELO3 = "meta-llama/llama-4-scout-17b-16e-instruct"


def recortar_y_escalar(ruta_imagen, top_pct, bottom_pct, escala=2, overlap_pct=0.05):
    img = Image.open(ruta_imagen)
    w, h = img.size
    top    = max(0, int(h * (top_pct - overlap_pct)))
    bottom = min(h, int(h * (bottom_pct + overlap_pct)))
    recorte = img.crop((0, top, w, bottom))
    recorte = recorte.resize((w * escala, (bottom - top) * escala), Image.LANCZOS)
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    OUTPUTS = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs"
    ruta_salida = os.path.join(OUTPUTS, f"{nombre}_trozo_{top_pct}_{bottom_pct}.png")
    recorte.save(ruta_salida)
    print(f"Guardado: {ruta_salida} — tamaño: {recorte.size}")
    return ruta_salida

INPUTS  = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\inputs"
CORTES = [(0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0)]

imagenes = [
    os.path.join(INPUTS, f)
    for f in os.listdir(INPUTS)
    if f.lower().endswith(".png")
]
print(f"Imágenes encontradas: {len(imagenes)}")

from docling.document_converter import DocumentConverter
converter = DocumentConverter()

dfs_trozos = []
for ruta_imagen in imagenes:
    print(f"\n{'='*50}\n📄 {os.path.basename(ruta_imagen)}")
    trozos = [recortar_y_escalar(ruta_imagen, top, bot) for top, bot in CORTES]
    for i, trozo in enumerate(trozos, 1):
        print(f"\n── Trozo {i} ──")
        result = converter.convert(trozo)
        print(f"  Tablas detectadas: {len(result.document.tables)}")
        for tabla in result.document.tables:
            df_trozo = tabla.export_to_dataframe()
            df_trozo = pd.DataFrame(
                [df_trozo.columns.tolist()] + df_trozo.values.tolist()
            )
            df_trozo.columns = [f"C{j}" for j in range(df_trozo.shape[1])]
            dfs_trozos.append(df_trozo)

Imágenes encontradas: 2

📄 excel_balance (limpio).png
Guardado: C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\excel_balance (limpio)_trozo_0_0.25.png — tamaño: (2924, 1778)
Guardado: C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\excel_balance (limpio)_trozo_0.25_0.5.png — tamaño: (2924, 2076)
Guardado: C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\excel_balance (limpio)_trozo_0.5_0.75.png — tamaño: (2924, 2076)


[INFO] 2026-05-07 22:18:04,193 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 22:18:04,197 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 22:18:04,198 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 22:18:04,292 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 22:18:04,294 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 22:18:04,295 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 22:18:04,339 [RapidOCR] base.py:22: Using engine_nam

Guardado: C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\excel_balance (limpio)_trozo_0.75_1.0.png — tamaño: (2924, 1780)

── Trozo 1 ──


[INFO] 2026-05-07 22:18:04,349 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.onnx
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 11005.30it/s]
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  Tablas detectadas: 1

── Trozo 2 ──


...

  Tablas detectadas: 1

── Trozo 3 ──


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  Tablas detectadas: 1

── Trozo 4 ──


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  Tablas detectadas: 1


## Celda 2 — Extracción de fechas canónicas con LLM multimodal

Las fechas de los ejercicios solo aparecen en la cabecera de la imagen (primeras ~250px de alto). La función `extraer_fechas_vision`:
1. Recorta únicamente esa franja superior de la imagen.
2. La codifica en base64.
3. La envía al modelo de visión `llama-4-scout-17b` con un prompt minimalista que pide exclusivamente las fechas en formato `DD/MM/AAAA`.

**Por qué usar visión solo para las fechas y no para todo el balance:**
Los modelos de visión gratuitos presentan errores sistemáticos al leer valores numéricos de tablas densas: confunden columnas, inventan partidas y asignan valores incorrectos (documentado en `experimentos_recortes.ipynb`). Sin embargo, son perfectamente fiables para leer texto de cabecera en un recorte pequeño y limpio. Se usa el mínimo indispensable de visión directa para minimizar el riesgo de error.

Las fechas detectadas aquí (`fechas_canonicas`) se usarán en el paso de layout para asignar correctamente cada columna de valores a su período.


In [5]:
import base64

from openai import OpenAI

client1 = OpenAI(
    api_key=GROQ_API_KEY1,
    base_url=GROQ_BASE_URL
)

MODELO3 = "meta-llama/llama-4-scout-17b-16e-instruct"

import base64
import tempfile
import os

def extraer_fechas_vision(ruta_imagen):
    img = Image.open(ruta_imagen)
    w, h = img.size
    header = img.crop((0, 0, w, min(250, h)))
    
    tmp_path = os.path.join(tempfile.gettempdir(), "header_fechas.png")
    header.save(tmp_path)
    with open(tmp_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()
    
    response = client1.chat.completions.create(
        model=MODELO3,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
                {"type": "text", "text": "Extrae todas las fechas que aparecen en esta imagen (formato DD/MM/AAAA). Devuelve SOLO un JSON así: {\"fechas\": [\"31/12/2024\", \"31/12/2023\"]}. Sin markdown."}
            ]
        }],
        temperature=0
    )
    raw = response.choices[0].message.content.strip().strip("```json").strip("```").strip()
    return json.loads(raw)["fechas"]

## Celda 3 — Ejecución: detección de fechas de la imagen de balance

Llama a `extraer_fechas_vision` sobre la primera imagen de la carpeta de inputs, que es la que contiene la cabecera con los períodos. El resultado (`fechas_canonicas`) se imprime para verificación antes de continuar.


In [6]:
# Extraer fechas de la primera imagen (el balance, que es donde están las fechas de cabecera)
fechas_canonicas = extraer_fechas_vision(r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\inputs\excel_balance (limpio).png")
print(f"Fechas detectadas: {fechas_canonicas}")

Fechas detectadas: ['31/12/2024', '31/12/2023', '31/12/2022']


## Celda 4 — Inspección de los DataFrames extraídos por Docling

Celda de diagnóstico: muestra todos los DataFrames de `dfs_trozos` para verificar visualmente que Docling ha separado correctamente las filas y que no quedan celdas fusionadas. 

Es útil ejecutar esta celda al cambiar la imagen de entrada para decidir si el número de recortes es suficiente: si se observan filas combinadas (una celda contiene el texto de dos o más partidas concatenadas), hay que aumentar el número de trozos en la celda 1.


In [7]:
#Esta celda es para revisar las tablas sacadas de cada trozo

for i, df in enumerate(dfs_trozos, 1):
    print(f"\n── Tabla {i} ──")
    display(df)


── Tabla 1 ──


,C0,C1,C2,C3,C4
0,,GRIFOLS SA,,,
1,Balance/Estadoderesultados,,,,
2,CuentasNoConsolidadas,31/12/2024,,,31/12/2022
3,,mil EUR,,,mil EUR
4,,,,12meses,
5,,Aprobado,Aprobado,Aprobado,
6,,NormalPGC2007,Normal PGC 2007,NormalPGC2007,
7,Activo,,,,
8,A)Activonocorriente,12497590,11416223,12647002,
9,IInmovilizadointangible,23959,19941,23213,



── Tabla 2 ──


,C0,C1,C2,C3
0,2.Creditosasociedadespuestasenequivalencia,n.d.,n.d.,n.d.
1,3.Otros activos financieros,n.d.,n.d.,n.d.
2,4.Otras inversiones,n.d.,n.d.,n.d.
3,VInversionesfinancierasalargoplazo,418896,2714,29199
4,VIActivosporimpuestodiferido,82263,49593,9150
5,VllDeudascomercialesnocorrientes,n.d.,n.d.,n.d.
6,B)Activo corriente,342446,1544994,246099
7,IActivosnocorrientesmantenidosparalaventa,n.d.,1360089,n.d.
8,IIExistencias,13342,12333,11439
9,IllDeudorescomercialesyotrascuentasacobrar,79885,79873,72869



── Tabla 3 ──


...

,C0,C1,C2,C3
0,b)Subvencionesdeexplotacionincorporadas alresu...,98,87,75
1,6.Gastos depersonal,-113208,-119602,-81088
2,"a)Sueldos,salariosyasimilados",-95399,-103507,-65742
3,b) Cargas sociales,-17809,-16095,-15194
4,c)Provisiones,n.d.,n.d.,-152
5,7.Otros gastos de explotacion,-294670,-269381,-214386
6,"a)Perdidas,deterioroyvariaciondeprovisionespor...",n.d.,n.d.,n.d.
7,b)Otrosgastosdegestioncorriente,n.d.,n.d.,n.d.
8,c)Gastosporemisi6ndegasesdeefectoinvemadero,n.d.,n.d.,n.d.
9,8.Amortizacion del inmovilizado,-16360,-17294,-14341



── Tabla 7 ──


,C0,C1,C2,C3
0,0,1,2,3
1,13.Diferencianegativaencombinacionesdenegocios,n.d.,n.d.,n.d.
2,14.Otros resultados,-172,-396,n.d.
3,A1)Resultado de explotaci6n (1 + 2+ 3+ 4+5 + 6...,518904,205348,58938
4,15. Ingresos financieros,10140,10219,8905
5,a)Departicipacioneseninstrumentosdepatrimonio,2060,n.d.,n.d.
6,b)Devaloresnegociablesyotrosinstrumentosfinanc...,8080,10219,8905
7,"c)Imputacionde subvenciones,donacionesylegados...",n.d.,n.d.,n.d.
8,16.Gastosfinancieros,-605691,-537309,-401985
9,17.Variaciondevalorrazonableeninstrumentosfina...,21596,2141,16689



── Tabla 8 ──


,C0,C1,C2,C3
0,b)Resultadosporenajenacionesy otras,n.d. n.d.,,n.d.
1,20.Otrosingresosygastosdecaracterfinanciero,308,620,368
2,a)Incorporacionalactivodegastosfinancieros,308,620,368
3,b)Ingresosfinancierosderivadosdeconveniosdeacr...,n.d.,n.d.,n.d.
4,c)Resto deingresosygastos,n.d.,n.d.,n.d.
5,A2）Resultadofinanciero（15+16+17+18+19+20),-594359,-523658,-366804
6,21.Participacionenbeneficios(perdidas)desocied...,n.d.,n.d.,n.d.
7,22.Deterioroyresultadosporenajenacionesdeparti...,n.d.,n.d.,n.d.
8,23.Diferencianegativadeconsolidaciondesociedad...,n.d.,n.d.,n.d.
9,A3)Resultadoantesdeimpuestos(A1+A2+21+22+23),-75455,-318310,-307866


## Celda 5 — Serialización del primer trozo para diagnóstico

Genera el texto serializado `fila=i, col=Cj: valor` del primer trozo (el que contiene la cabecera con fechas y metadatos). Se imprime para verificar que la estructura es legible antes de enviársela al LLM de layout.

El primer trozo es el más informativo para la detección de layout porque contiene la cabecera completa; los trozos siguientes tienen la misma estructura de columnas pero sin ella.


In [8]:
#Este es el mapeo que se le envia al llm para que detecte el layout osea vea en que columnas estan los valores o las partidas por si cambiara o algo

df = dfs_trozos[0]

lineas = []
for row_idx, row in df.head(15).iterrows():
    for col_name, val in row.items():
        if pd.notna(val) and str(val).strip() not in ("", "nan"):
            lineas.append(f"fila={row_idx}, col={col_name}: {val}")

mapa = "\n".join(lineas)
print(mapa)

fila=0, col=C1: GRIFOLS SA
fila=1, col=C0: Balance/Estadoderesultados
fila=2, col=C0: CuentasNoConsolidadas
fila=2, col=C1: 31/12/2024
fila=2, col=C4: 31/12/2022
fila=3, col=C1: mil EUR
fila=3, col=C4: mil EUR
fila=4, col=C3: 12meses
fila=5, col=C1: Aprobado
fila=5, col=C2: Aprobado
fila=5, col=C3: Aprobado
fila=6, col=C1: NormalPGC2007
fila=6, col=C2: Normal PGC 2007
fila=6, col=C3: NormalPGC2007
fila=7, col=C0: Activo
fila=8, col=C0: A)Activonocorriente
fila=8, col=C1: 12497590
fila=8, col=C2: 11416223
fila=8, col=C3: 12647002
fila=9, col=C0: IInmovilizadointangible
fila=9, col=C1: 23959
fila=9, col=C2: 19941
fila=9, col=C3: 23213
fila=10, col=C0: 1.Fondodecomerciodeconsolidacion
fila=10, col=C1: n.d.
fila=10, col=C2: n.d.
fila=10, col=C3: n.d.
fila=11, col=C0: 2. Investigaci6n
fila=11, col=C1: n.d.
fila=11, col=C2: n.d.
fila=11, col=C3: n.d.
fila=12, col=C0: 3.Propiedad intelectual
fila=12, col=C1: n.d.
fila=12, col=C2: n.d.
fila=12, col=C3: n.d.
fila=13, col=C0: 5.Otro inmovilizado

## Celda 6 — Paso de layout: detección por trozo (llamada individual)

Para cada trozo se serializa su contenido y se envía al LLM con un prompt que incluye las `fechas_canonicas` ya conocidas. El modelo devuelve un JSON con:
- `columna_partidas`: columna de Docling donde están los nombres de las partidas.
- `columnas_valores`: lista de columnas con valores numéricos, en el mismo orden que las fechas.
- `unidades`: unidad monetaria detectada.

**Rotación de clientes Groq:** se usan 3 clientes con distintas API keys rotando en round-robin (`clients[i % len(clients)]`). Esto es necesario porque el plan gratuito de Groq limita a 30 RPM por key, y con 8 trozos se superaría ese límite en una sola ejecución.

Los layouts de todos los trozos quedan almacenados en el diccionario `layouts` indexado por posición de trozo.


In [9]:
import os
from dotenv import load_dotenv
import openai
import json

load_dotenv()

GROQ_BASE_URL = "https://api.groq.com/openai/v1"

clients = [
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_1",   ""), base_url=GROQ_BASE_URL),
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_2", ""), base_url=GROQ_BASE_URL),
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_3", ""), base_url=GROQ_BASE_URL),
]

layouts = {}

for i, df in enumerate(dfs_trozos):
    print(f"\n{'='*50}")
    print(f"── Trozo {i+1}/{len(dfs_trozos)} ──")

    client = clients[i % len(clients)]  # rota entre los 3 clientes

    lineas = []
    for row_idx, row in df.head(15).iterrows():
        for col_name, val in row.items():
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                lineas.append(f"fila={row_idx}, col={col_name}: {val}")

    mapa = "\n".join(lineas)

    prompt = f"""Tienes un fragmento de un balance financiero extraído con OCR, mapeado celda a celda.
Cada línea indica exactamente en qué fila y columna está cada valor no vacío.

{mapa}

IMPORTANTE: Las fechas de este balance son exactamente estas en orden: {fechas_canonicas}

Analiza la estructura y responde SOLO en JSON sin markdown ni backticks:

{{
  "unidades": "mil EUR / EUR / desconocido",
  "columna_partidas": "columna exacta con los nombres de las partidas",
  "columnas_valores": ["columnas con valores numéricos"],
  "razonamiento": "breve explicación",
  "ejemplo": {{
    "partida": "nombre de partida de ejemplo",
    "valores": {{"nombre_columna": valor_numerico}}
  }}
}}

Instrucciones clave:
- Los valores numéricos son importes grandes del balance (ej: 12497590, 11416223).
- La columna de partidas contiene descripciones contables: 'Activo', 'Inmovilizado', 'Existencias', etc.
- Puede haber columnas intermedias vacías o con metadatos, ignóralas.
- Fíjate bien en qué columna aparecen los números grandes para identificar columnas_valores.
"""

    response = client.chat.completions.create(
        model=MODELO1,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip().strip("```json").strip("```").strip()

    try:
        layout = json.loads(raw)
        layouts[i] = layout
        print(f"Unidades:         {layout['unidades']}")
        print(f"Columna partidas: {layout['columna_partidas']}")
        print(f"Columnas valores: {layout['columnas_valores']}")
        print(f"Ejemplo:          {layout['ejemplo']}")
    except json.JSONDecodeError:
        print(f"⚠️ No devolvió JSON limpio, revisa raw:")
        print(raw)


── Trozo 1/8 ──
Unidades:         mil EUR / desconocido
Columna partidas: C0
Columnas valores: ['C1', 'C2', 'C3']
Ejemplo:          {'partida': 'A)Activonocorriente', 'valores': {'C1': 12497590, 'C2': 11416223, 'C3': 12647002}}

── Trozo 2/8 ──
Unidades:         EUR
Columna partidas: C0
Columnas valores: ['C1', 'C2', 'C3']
Ejemplo:          {'partida': 'VInversionesfinancierasalargoplazo', 'valores': {'C1': 418896, 'C2': 2714, 'C3': 29199}}

── Trozo 3/8 ──
Unidades:         EUR
Columna partidas: C0
Columnas valores: ['C1', 'C2', 'C3']
Ejemplo:          {'partida': 'A-1)Fondospropios', 'valores': {'C1': 1972243, 'C2': 2044735, 'C3': 2285248}}

── Trozo 4/8 ──
Unidades:         mil EUR
Columna partidas: C0
Columnas valores: ['C1', 'C2', 'C3']
Ejemplo:          {'partida': 'IlI Deudas con empresas del grupo y asociadas al largo plazo', 'valores': {'C1': 4276854, 'C2': 4673555, 'C3': 6419171}}

── Trozo 5/8 ──
Unidades:         EUR
Columna partidas: C0
Columnas valores: ['C1', 'C2', 'C3'

## Construcción del DataFrame unificado


## Celda 7 — Construcción del DataFrame de conceptos y exportación a CSV

Con el layout detectado para cada trozo se recorre cada DataFrame de `dfs_trozos`, se extrae la columna de partidas y las columnas de valores, y se asocia cada valor a su fecha canónica mediante el mapeo `col_valor → fecha`. El resultado es un DataFrame largo con columnas `{partida, fecha, valor, trozo}`.

Se exporta a CSV como paso intermedio para facilitar la inspección manual y la depuración.

> **Diferencia respecto al pipeline Excel:** aquí los nombres de partidas pueden contener ruido OCR (letras pegadas, signos de puntuación intercalados, palabras concatenadas sin espacio). Esto es manejable porque el LLM de selección de partidas del siguiente paso está instruido explícitamente para tolerarlo.


In [10]:
registros = []

for i, df in enumerate(dfs_trozos):
    layout = layouts[i]
    col_partidas = layout['columna_partidas']
    cols_valores = layout['columnas_valores']

    col_a_fecha = {cv: cf for cv, cf in zip(cols_valores, fechas_canonicas)}

    for _, row in df.iterrows():
        partida = str(row.get(col_partidas, "")).strip()
        if not partida or partida in ("nan", ""):
            continue
        for col_val, fecha in col_a_fecha.items():
            if col_val not in df.columns:
                continue
            val = str(row.get(col_val, "")).strip()
            try:
                val = float(val)
            except ValueError:
                val = None
            registros.append({"partida": partida, "fecha": fecha, "valor": val, "trozo": i+1})

df_resultado = pd.DataFrame(registros)
pd.set_option('display.max_rows', None)
display(df_resultado)
pd.reset_option('display.max_rows')
df_resultado.to_csv(r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\balance_final.csv", index=False)

,partida,fecha,valor,trozo
0,Balance/Estadoderesultados,31/12/2024,NaN,1
1,Balance/Estadoderesultados,31/12/2023,NaN,1
2,Balance/Estadoderesultados,31/12/2022,NaN,1
3,CuentasNoConsolidadas,31/12/2024,NaN,1
4,CuentasNoConsolidadas,31/12/2023,NaN,1
5,CuentasNoConsolidadas,31/12/2022,NaN,1
6,Activo,31/12/2024,NaN,1
7,Activo,31/12/2023,NaN,1
8,Activo,31/12/2022,NaN,1
9,A)Activonocorriente,31/12/2024,12497590.0,1


## Celda 8 — Selección de las 22 partidas mediante LLM con índice posicional

Mismo enfoque que en el pipeline Excel pero con las instrucciones especiales adaptadas al ruido OCR típico de imágenes:
- Las partidas pueden estar truncadas o con palabras pegadas (`"ACTIVOSCORRIENTES"`, `"Deudorescomercialesyotrascuentasacobrar"`).
- El prompt menciona explícitamente variantes OCR esperadas para las partidas más conflictivas.
- Para `deuda_credito_lp` y `deuda_credito_cp` el razonamiento por anclas LP/CP sigue siendo la clave para resolver la ambigüedad de partidas con nombre idéntico en secciones distintas del balance.

El output es un JSON `{alias: {idx, partida}}` con los 22 mapeos sobre la lista de partidas únicas del DataFrame.


In [11]:
# ── PASO 3: LLM selecciona partidas directamente ──────────────────────────
partidas_unicas = df_resultado['partida'].drop_duplicates().tolist()
partidas_formateadas = "\n".join(f"idx={i}: {p}" for i, p in enumerate(partidas_unicas))

prompt_mapeo = f"""Eres un experto en contabilidad española (PGC 2007).
Tienes la lista completa de partidas de un balance financiero extraído con OCR,
en orden exacto tal como aparecen (los textos pueden estar truncados o con ruido OCR).

PARTIDAS DISPONIBLES:
{partidas_formateadas}

INSTRUCCIONES GENERALES:
- Devuelve el idx y el texto EXACTAMENTE como aparecen en la lista.
- Si una partida no existe en este balance devuelve null.

INSTRUCCIONES ESPECIALES:
- 'patrimonio_neto': puede aparecer como "Patrimonio neto", con letras/números antes o después, o todo junto.
- 'fondos_propios': si no hay partida específica usa patrimonio_neto para los dos; si hay fondos propios devuelve la partida específica.
- 'pasivo_no_corriente': puede aparecer como 'B) Pasivo no corriente'.
- 'pasivo_corriente': puede aparecer como 'C) Pasivo corriente'.
- 'total_activo': puede aparecer como 'Total activo (A + B)'.
- 'activo_corriente': puede aparecer como 'B) Activo corriente'.
- 'resultado_ejercicio': puede aparecer como 'A5) Resultado del ejercicio'.
- 'efectivo': puede aparecer como 'VII Efectivo y otros activos liquidos equivalentes'.
- 'deuda_credito_lp' y 'deuda_credito_cp': busca "Deudas con entidades de credito".
  Puede aparecer dos veces. Mira los índices anteriores para determinar si está más cerca
  de anclas LP ("Deudas a largo plazo", "largo plazo", "pasivo no corriente") o
  anclas CP ("Deudas a corto plazo", "corto plazo", "pasivo corriente").
  Asigna según la ancla más cercana en número de índice.

Devuelve SOLO JSON sin markdown, con este formato:
{{
  "patrimonio_neto": {{"idx": 40, "partida": "texto literal"}},
  "fondos_propios": {{"idx": 41, "partida": "texto literal"}},
  "pasivo_no_corriente": {{"idx": 61, "partida": "texto literal"}},
  "deudas_lp": {{"idx": 63, "partida": "texto literal"}},
  "deudas_cp": {{"idx": 80, "partida": "texto literal"}},
  "pasivo_corriente": {{"idx": 75, "partida": "texto literal"}},
  "total_activo": {{"idx": 46, "partida": "texto literal"}},
  "activo_corriente": {{"idx": 27, "partida": "texto literal"}},
  "existencias": {{"idx": 29, "partida": "texto literal"}},
  "deudores_comerciales": {{"idx": 31, "partida": "texto literal"}},
  "efectivo": {{"idx": 45, "partida": "texto literal"}},
  "inmovilizado_material": {{"idx": 14, "partida": "texto literal"}},
  "acreedores_comerciales": {{"idx": 88, "partida": "texto literal"}},
  "cifra_negocios": {{"idx": 100, "partida": "texto literal"}},
  "otros_ingresos_explotacion": {{"idx": 110, "partida": "texto literal"}},
  "resultado_explotacion": {{"idx": 132, "partida": "texto literal"}},
  "amortizacion": {{"idx": 121, "partida": "texto literal"}},
  "gastos_financieros": {{"idx": 137, "partida": "texto literal"}},
  "resultado_antes_impuestos": {{"idx": 155, "partida": "texto literal"}},
  "resultado_ejercicio": {{"idx": 160, "partida": "texto literal"}},
  "deuda_credito_lp": null,
  "deuda_credito_cp": {{"idx": 82, "partida": "texto literal"}}
}}
"""

response = client.chat.completions.create(
    model=MODELO2,
    messages=[{"role": "user", "content": prompt_mapeo}],
    temperature=0
)

raw = response.choices[0].message.content.strip().strip("```json").strip("```").strip()

try:
    mapeo_llm = json.loads(raw)
    print("\n── Mapeo LLM ──")
    for alias, info in mapeo_llm.items():
        if info:
            print(f"  {alias}: idx={info['idx']} → {info['partida']}")
        else:
            print(f"  {alias}: null")
except json.JSONDecodeError:
    print("⚠️ No devolvió JSON limpio")
    print(raw)


── Mapeo LLM ──
  patrimonio_neto: idx=45 → A)Patrimonioneto
  fondos_propios: idx=46 → A-1) Fondos propios
  pasivo_no_corriente: idx=73 → B)Pasivonocorriente
  deudas_lp: idx=77 → 2.Deudasconentidadesdecredito
  deudas_cp: idx=101 → 2.Deudasconentidadesdecredito
  pasivo_corriente: idx=87 → C)Pasivo corriente
  total_activo: idx=43 → Total activo (A + B)
  activo_corriente: idx=22 → B)Activo corriente
  existencias: idx=24 → IlExistencias
  deudores_comerciales: idx=25 → IlIDeudorescomercialesyotrascuentasacobrar
  efectivo: idx=42 → VllEfectivoyotrosactivoslfquidosequivalentes
  inmovilizado_material: idx=9 → IInmovilizadomaterial
  acreedores_comerciales: idx=104 → VAcreedorescomercialesyotrascuentasapagar
  cifra_negocios: idx=115 → A)Operacionescontinuadas
  otros_ingresos_explotacion: idx=125 → 5.Otrosingresosdeexplotacion
  resultado_explotacion: idx=149 → A1)Resultado de explotaci6n(1+2+3+4+5+6+7+8+9+10+11+12+13+14)
  amortizacion: idx=138 → 8.Amortizacion del inmovilizado
  

## Celda 9 — Extracción de valores por período

Con el mapeo del LLM se recupera para cada alias el valor numérico por fecha cruzando `partidas_unicas[info['idx']]` con `df_resultado`. El resultado es `valores_por_fecha`, idéntico en estructura al del pipeline Excel.


In [12]:
# ── PASO 4: extraer valores por periodo ──────────────────────────────────

valores_por_fecha = {}

for alias, info in mapeo_llm.items():
    if info is None:
        continue
    partida_real = partidas_unicas[info['idx']]
    filas = df_resultado[df_resultado['partida'] == partida_real]
    for _, row in filas.iterrows():
        fecha = row['fecha']
        valor = row['valor']
        if fecha not in valores_por_fecha:
            valores_por_fecha[fecha] = {}
        if alias not in valores_por_fecha[fecha]:
            valores_por_fecha[fecha][alias] = None if pd.isna(valor) else valor

print(pd.DataFrame(valores_por_fecha).T)

            patrimonio_neto  fondos_propios  pasivo_no_corriente  deudas_lp  \
31/12/2024        1953852.0       1972243.0           10572251.0   901345.0   
31/12/2023        2101487.0       2044735.0           10561625.0  1308026.0   
31/12/2022        2343125.0       2285248.0           10323483.0  1340473.0   

            deudas_cp  pasivo_corriente  total_activo  activo_corriente  \
31/12/2024    62437.0          313933.0    12840036.0          342446.0   
31/12/2023    64699.0          298105.0    12961217.0         1544994.0   
31/12/2022    61720.0          226493.0    12893101.0          246099.0   

            existencias  deudores_comerciales  ...  acreedores_comerciales  \
31/12/2024      13342.0               79885.0  ...                 94624.0   
31/12/2023      12333.0               79873.0  ...                112436.0   
31/12/2022      11439.0               72869.0  ...                 82987.0   

            cifra_negocios  otros_ingresos_explotacion  resultado_exp

## Celda 10 — Cálculo de 22 ratios financieros

Código idéntico al paso 5 del pipeline Excel. Los ratios se calculan sobre `valores_por_fecha` y se muestran en un DataFrame por período.

Los resultados son **idénticos a los del pipeline Excel** para los tres ejercicios analizados, lo que valida que el pipeline de imagen produce la misma extracción numérica que la lectura directa del fichero estructurado.

> **Nota sobre la amortización:** en SABI (Excel) la amortización viene en valor absoluto, por lo que `EBITDA = EBIT - amort` es directo. En imágenes exportadas desde SABI el signo también es positivo al proceder del mismo fichero fuente.


In [13]:
# ── PASO 5: calcular ratios ───────────────────────────────────────────────

def safe_div(a, b):
    if a is None or b is None or b == 0: return None
    return round(a / b, 6)

def pct(a, b):   r = safe_div(a, b); return round(r * 100, 2) if r is not None else None
def ratio(a, b): r = safe_div(a, b); return round(r, 2)       if r is not None else None
def dias(a, b):  r = safe_div(a, b); return round(r * 365, 1) if r is not None else None

ratios_calculados = {}

for fecha, v in valores_por_fecha.items():
    cn    = v.get('cifra_negocios')
    oi    = v.get('otros_ingresos_explotacion') or 0
    ebit  = v.get('resultado_explotacion')
    amort = v.get('amortizacion') or 0
    gf    = v.get('gastos_financieros')
    res   = v.get('resultado_ejercicio')
    act   = v.get('total_activo')
    actc  = v.get('activo_corriente')
    exst  = v.get('existencias') or 0
    deud  = v.get('deudores_comerciales')
    efec  = v.get('efectivo') or 0
    fp    = v.get('fondos_propios')
    pasc  = v.get('pasivo_corriente')
    acr   = v.get('acreedores_comerciales')
    delp  = v.get('deuda_credito_lp') or 0
    decp  = v.get('deuda_credito_cp') or 0
    pnc   = v.get('pasivo_no_corriente')

    ebitda  = (ebit - amort)  if ebit  is not None else None
    gf_abs  = abs(gf)         if gf    is not None else None
    pas_tot = (pnc + pasc)    if (pnc  is not None and pasc is not None) else None
    fm      = (actc - pasc)   if (actc is not None and pasc is not None) else None
    deuda_f = delp + decp
    dfn     = deuda_f - efec

    pmc   = dias(deud, cn)
    pmp   = dias(acr,  cn)
    stock = dias(exst, cn)
    ccc   = round((pmc or 0) + (stock or 0) - (pmp or 0), 1) if all(x is not None for x in [pmc, pmp, stock]) else None

    ratios_calculados[fecha] = {
        # Solvencia
        "SOL01_solvencia":              ratio(act, pas_tot),
        "SOL02_autonomia_financiera":   pct(fp, act),
        "SOL03_endeudamiento":          ratio(pas_tot, fp),
        "SOL04_dfn_sobre_fp":           pct(dfn, fp),
        # Cobertura
        "COV01_dfn_sobre_ebitda":       ratio(dfn, ebitda),
        "COV02_icr":                    ratio(ebitda, gf_abs),
        "COV03_carga_financiera_cn":    pct(gf_abs, cn),
        # Liquidez
        "LIQ01_ratio_corriente":        ratio(actc, pasc),
        "LIQ02_ratio_acido":            ratio((actc - exst) if actc is not None else None, pasc),
        "LIQ03_fm_sobre_cn":            pct(fm, cn),
        "LIQ04_tesoreria_sobre_activo": pct(efec, act),
        # Rentabilidad
        "REN01_margen_ebitda":          pct(ebitda, cn),
        "REN02_margen_ebit":            pct(ebit, cn),
        "REN03_margen_neto":            pct(res, cn),
        "REN04_roa":                    pct(ebit, act),
        "REN05_roe":                    pct(res, fp),
        # Apalancamiento
        "APL01_deuda_fin_sobre_activo": pct(deuda_f, act),
        "APL02_pasivo_sobre_activo":    pct(pas_tot, act),
        # Eficiencia
        "EFI01_pmc":   pmc,
        "EFI02_pmp":   pmp,
        "EFI03_stock": stock,
        "EFI04_ccc":   ccc,
    }

df_ratios = pd.DataFrame(ratios_calculados).T
df_ratios

,SOL01_solvencia,SOL02_autonomia_financiera,SOL03_endeudamiento,SOL04_dfn_sobre_fp,COV01_dfn_sobre_ebitda,COV02_icr,COV03_carga_financiera_cn,LIQ01_ratio_corriente,LIQ02_ratio_acido,LIQ03_fm_sobre_cn,...,REN02_margen_ebit,REN03_margen_neto,REN04_roa,REN05_roe,APL01_deuda_fin_sobre_activo,APL02_pasivo_sobre_activo,EFI01_pmc,EFI02_pmp,EFI03_stock,EFI04_ccc
31/12/2024,1.18,15.36,5.52,47.13,1.74,0.88,86.40,1.09,1.05,4.07,...,74.02,-11.86,4.04,-4.22,7.25,84.78,41.6,49.3,6.9,-0.8
31/12/2023,1.19,15.78,5.31,66.69,6.13,0.41,86.77,5.18,5.14,201.36,...,33.16,-39.84,1.58,-12.07,10.62,83.79,47.1,66.3,7.3,-11.9
31/12/2022,1.22,17.72,4.62,60.72,18.94,0.18,82.27,1.09,1.04,4.01,...,12.06,-54.50,0.46,-11.65,10.87,81.83,54.4,62.0,8.5,0.9


---

## Variante: detección de layout en llamada única

Las celdas siguientes implementan una optimización que reduce el consumo de tokens del paso de layout de **N llamadas individuales a 1 sola llamada**, enviando fingerprints compactos de todos los trozos juntos.

**Resultado del experimento:** para 8 trozos, pasar de 8 llamadas (~11.500 tokens) a 1 llamada unificada (~2.967 tokens) supone una reducción del **74% en tokens** con calidad equivalente en el mapeo de columnas.


## Celda 11 — Layout unificado: construcción del mapa compacto y llamada única

Se construye un texto con fingerprints de todos los trozos:
- Del trozo 1 (que tiene cabecera): 20 filas para que el modelo vea las fechas y metadatos.
- De los trozos siguientes: solo 5 filas, suficiente para identificar las columnas de valores.

El prompt pide un JSON con la estructura de columnas de **todos los trozos a la vez**, reduciendo el número de llamadas de N a 1. Se reportan los tokens consumidos para comparar con el enfoque anterior.


In [14]:
bloques = []
for i, df in enumerate(dfs_trozos):
    n_filas = 20 if i == 0 else 5
    
    lineas = []
    for row_idx, row in df.head(n_filas).iterrows():
        for col_name, val in row.items():
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                lineas.append(f"f{row_idx},c{col_name}:{val}")
    
    bloques.append(f"[TROZO {i+1}]\n" + " | ".join(lineas))

mapa_global = "\n".join(bloques)

prompt = f"""Tienes {len(dfs_trozos)} fragmentos de estados financieros extraídos con OCR.
Fechas del balance en orden: {fechas_canonicas}
Los valores numéricos son importes grandes (ej: 12497590, 418896). Los "n.d." son celdas sin dato.
La columna de partidas contiene descripciones contables (Activo, Pasivo, Existencias...).
El TROZO 1 contiene la cabecera completa con fechas y metadatos.
Los trozos siguientes comparten la misma estructura de columnas que el TROZO 1 pero sin cabecera.
{mapa_global}
Devuelve SOLO este JSON sin markdown:
{{
  "unidades": "mil EUR / EUR / desconocido",
  "trozos": [
    {{"trozo": 1, "col_partidas": "C0", "cols_valores": ["C1","C2","C3"]}},
    {{"trozo": 2, "col_partidas": "C0", "cols_valores": ["C1","C2","C3"]}},
    ...
  ]
}}"""

response = clients[0].chat.completions.create(
    model=MODELO1,
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

input_tokens  = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
total_tokens  = response.usage.total_tokens

print(f"Input tokens:  {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Total tokens:  {total_tokens}")

raw = response.choices[0].message.content.strip().strip("```json").strip("```").strip()
layout_global = json.loads(raw)
print(layout_global)

Input tokens:  2522
Output tokens: 445
Total tokens:  2967
{'unidades': 'mil EUR / EUR / desconocido', 'trozos': [{'trozo': 1, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 2, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 3, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 4, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 5, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 6, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 7, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}, {'trozo': 8, 'col_partidas': 'C0', 'cols_valores': ['C1', 'C2', 'C3']}]}


## Celda 12 — Construcción del DataFrame con layout unificado

Mismo proceso que la celda 7 pero usando el layout del JSON unificado (`layout_global`). El resultado final es equivalente, confirmando que la optimización de tokens no penaliza la calidad de extracción.


In [15]:
# Construir lookup: trozo_idx -> {col_partidas, cols_valores}
lookup = {t["trozo"] - 1: t for t in layout_global["trozos"]}

registros = []

for i, df in enumerate(dfs_trozos):
    layout = lookup.get(i)
    if layout is None:
        print(f"⚠️ No hay layout para trozo {i+1}, saltando")
        continue

    col_partidas = layout["col_partidas"]
    cols_valores = layout["cols_valores"]

    col_a_fecha = {cv: cf for cv, cf in zip(cols_valores, fechas_canonicas)}

    for _, row in df.iterrows():
        partida = str(row.get(col_partidas, "")).strip()
        if not partida or partida in ("nan", ""):
            continue
        for col_val, fecha in col_a_fecha.items():
            if col_val not in df.columns:
                continue
            val = str(row.get(col_val, "")).strip()
            try:
                val = float(val)
            except ValueError:
                val = None
            registros.append({"partida": partida, "fecha": fecha, "valor": val, "trozo": i+1})

df_resultado = pd.DataFrame(registros)
pd.set_option('display.max_rows', None)
display(df_resultado)
pd.reset_option('display.max_rows')
df_resultado.to_csv(r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\outputs\balance_final.csv", index=False)

,partida,fecha,valor,trozo
0,Balance/Estadoderesultados,31/12/2024,NaN,1
1,Balance/Estadoderesultados,31/12/2023,NaN,1
2,Balance/Estadoderesultados,31/12/2022,NaN,1
3,CuentasNoConsolidadas,31/12/2024,NaN,1
4,CuentasNoConsolidadas,31/12/2023,NaN,1
5,CuentasNoConsolidadas,31/12/2022,NaN,1
6,Activo,31/12/2024,NaN,1
7,Activo,31/12/2023,NaN,1
8,Activo,31/12/2022,NaN,1
9,A)Activonocorriente,31/12/2024,12497590.0,1
